# EVALUATIONS
There two types of evaluations covered in this notebook:
1. retrieval evaluations
2. generative evaluations

## Retrieval evaluation
### Accuracy@k

Our most fundamental metric, measuring the presence of at least one relevant document in the top-k results for each query. While simple, it provides essential validation of basic retrieval capability. An Accuracy@k of 0.90 indicates that 90% of queries successfully retrieved at least one relevant document within their top k results.

Technical implementation:

$Accuracy@k = (queries with ≥1 relevant doc in top k) / (total queries)$

### NDCG@k (Normalized Discounted Cumulative Gain)

NDCG captures both the presence and positioning of relevant documents in ranked results. The key insight is that relevant documents appearing lower in the ranking contribute diminishing value to the overall score. The normalization against an ideal ranking produces a score between 0 and 1, enabling comparison across different queries and result sets.

Technical implementation:

$DCG@k = Σ(i=1 to k) rel_i / log2(i + 2)$

$NDCG@k = DCG@k / IDCG@k$

Where rel_i represents the relevance (0 or 1) of the document at position i, and IDCG represents the DCG of a perfect ranking. 

### Precision@k and Recall@k

These complementary metrics evaluate retrieval effectiveness from different perspectives:

Precision@k measures result set accuracy by calculating the fraction of relevant documents among the top k results. A Precision@5 of 0.8 indicates that 4 of the top 5 results were relevant.

Recall@k quantifies retrieval completeness by measuring the fraction of all relevant documents found within the top k results. A Recall@10 of 0.7 indicates that 70% of all relevant documents appear in the top 10 results.

Technical implementation:

$Precision@k = (relevant docs in top k) / k$

$Recall@k = (relevant docs in top k) / (total relevant docs)$

### Mean Reciprocal Rank (MRR@k)

MRR focuses specifically on the position of the first relevant document in the ranking. The reciprocal rank for a query is 1/position of the first relevant result, with the final metric averaged across all queries. This is particularly valuable for evaluating systems where the position of the first relevant result is critical.

Technical implementation:

$MRR = (1/|Q|) Σ(i=1 to |Q|) 1/rank_i$

### Mean Average Precision (MAP@k)

MAP provides a comprehensive single-score assessment of ranking quality. It incorporates both the precision at each relevant document position and the total recall, making it particularly effective for evaluating overall retrieval performance.

Technical implementation:

$AP@k = (1/min(k, R)) Σ(r=1 to k) (P@r * rel(r))$

$MAP@k = (1/|Q|) Σ(q=1 to |Q|) AP@k(q)$

Where:

R represents total relevant documents

P@r is precision at rank r

rel(r) is 1 for relevant results, 0 otherwise

|Q| represents the total number of queries



# Retrieval evaluation

In [1]:
import pandas as pd
import numpy as np
import pickle
from typing import List, Dict

In [2]:
with open("../data/queries.pkl","rb") as f:
    queries=pickle.load(f)

In [3]:
with open("../data/relevant_docs.pkl","rb") as f:
    relevant_docs=pickle.load(f)

In [4]:
with open("../data/corpus_all.pkl","rb") as f:
    corpus = pickle.load(f)

In [5]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator,SequentialEvaluator
from sentence_transformers.util import cos_sim

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

import pickle

from typing import Dict

collection_name='simple_rag'

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

en_dim=len(embeddings.embed_query("blah"))

def get_qdrant_client():
    """Create a singleton Qdrant client."""
    return QdrantClient("http://localhost:6333")

with open('../data/corpus_all.pkl','rb') as f:
    chunks = pickle.load(f)

client=get_qdrant_client()

def get_qdrant_collection():
    try:
        client.get_collection(collection_name=collection_name)
    except Exception as e:
        print(f"e\ncollection  {collection_name} does not exitst")
        print(f"create collection {collection_name} . . .")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=en_dim, distance=Distance.COSINE),
        )
        print(f"collection {collection_name} created")

def create_qdrant_point(chunks:Dict):
    print(f"start create {len(chunks.keys())} chunks and qdrant data point creation . . . ")
    Points =[
            PointStruct(
                id = key,
                vector = np.array(embeddings.embed_query(chunk)),
                payload = {
                    "text" : chunk,
                    "book" : "the republic",
                    "file_path" : "data/the_republic_introduction.txt"
                    },
                )
           for key,chunk in chunks.items()
    ]
    #upsert to qdrant
    print(f"{len(Points)} points created . . . ")
    return Points

def load_qdrant(Points:List, BATCH=1000):
    total_count=len(Points)
    current_count=0
    while True:
        if current_count + BATCH >= total_count:
            client.upsert(collection_name=collection_name, points=Points[current_count:])
            current_count=total_count
            print(f"total point loaded: {current_count}")
            break
        elif current_count == 0:
            client.upsert(collection_name=collection_name, points=Points[:BATCH])
            current_count=current_count+BATCH
        else:
            client.upsert(collection_name=collection_name, points=Points[current_count:current_count+BATCH])
            current_count=current_count+BATCH
    print(f"total point loaded: {current_count}")

In [7]:
def semantic_search(
    query: str, collection_name: str, top_k: int = 5
) -> List[Dict]:
    """Perform semantic search on code chunks."""
    contexts=[]
    qry_vec = embeddings.embed_query(query)
    client = get_qdrant_client()

    try:
        results = client.query_points(
            collection_name=collection_name, query=np.array(qry_vec), limit=top_k, score_threshold = 0.7
        )
        return [
            {
                "id": hit.id,
                "file_path": hit.payload["file_path"],
                "book": hit.payload["book"],
                "text": hit.payload["text"],
                "score": hit.score,
            }
            for hit in results.points
        ]
    except Exception as e:
        print(f"Search error: {e}")
        return []

In [27]:
relevant_docs2={}
retrieved_contexts=[]
for key in queries.keys():
    results = semantic_search(queries[key],collection_name=collection_name,top_k=10)
    docs = [ res['id'] for res in results ]
    retrieved_contexts.append([ res['text'] for res in results ])
    relevant_docs2[key]=docs

In [9]:
# Dimensions of interest
matryoshka_dimensions = [768, 512, 256, 128, 64] # Important: large to small

# Create empty list to hold evaluators
matryoshka_evaluators = []

for dim in matryoshka_dimensions:
    ir_evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=f"dim_{dim}",
        truncate_dim=dim,  # Truncate the embeddings to the respective dimension
        score_functions={"cosine": cos_sim},
    )
    matryoshka_evaluators.append(ir_evaluator)

# Create a sequential evaluator
# Able to run all our dimension specific InformationRetrievalEvaluators sequentially.

In [10]:
evaluator = SequentialEvaluator(matryoshka_evaluators)

In [14]:
model=SentenceTransformer("all-MiniLM-L6-v2")

In [15]:
base_results = evaluator(model)

In [16]:
# Print header
print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'map@100',
    'accuracy@1',
    'accuracy@3',
    'accuracy@5',
    'accuracy@10',
    'precision@1',
    'precision@3',
    'precision@5',
    'precision@10',
    'recall@1',
    'recall@3',
    'recall@5',
    'recall@10'
]

# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dimensions:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(base_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {base_results['sequential_score']:1f}")


Base Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            1.0000       1.0000       1.0000       0.9855       0.9287 
mrr@10                 1.0000       1.0000       1.0000       0.9853       0.9201 
map@100                1.0000       1.0000       1.0000       0.9779       0.8894 
accuracy@1             1.0000       1.0000       1.0000       0.9706       0.8676 
accuracy@3             1.0000       1.0000       1.0000       1.0000       0.9706 
accuracy@5             1.0000       1.0000       1.0000       1.0000       1.0000 
accuracy@10            1.0000       1.0000       1.0000       1.0000       1.0000 
precision@1            1.0000       1.0000       1.0000       0.9706       0.8676 
precision@3            0.5196       0.5196       0.5

# generative evaluations

## Data prep

In [1]:
import os
from openai import OpenAI

LLM_SERVER='https://openrouter.ai/api/v1'
LLM_API_KEY='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' #os.environ["OPENROUTER_API_KEY"]
LLM_MODEL='x-ai/grok-4.1-fast:free'

client = OpenAI(base_url=LLM_SERVER,api_key=LLM_API_KEY)
LOCAL=True

In [18]:
import instructor
from openai import AsyncOpenAI,OpenAI
from ragas.llms import llm_factory
#from langchain_community.llms import OpenLLM

LOCAL=True

if LOCAL:
    #llm = OpenLLM(base_url="http://localhost:8081/v1", api_key="na") 
    client = AsyncOpenAI(base_url="http://localhost:8081/v1", api_key="na")
    llm = llm_factory("OpenAI",client=client)
else:
    client = AsyncOpenAI(base_url=LLM_SERVER,api_key=LLM_API_KEY)
    llm=llm_factory(LLM_MODEL,client=client)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [19]:
from ragas.metrics.collections import Faithfulness

metric = Faithfulness(llm)
result = await metric.ascore(user_input="Where was Einstein born?",\
    response="Einstein was born in Germany on 14th March 1879.",\
    retrieved_contexts=["Albert Einstein was born in Germany..."])

print(f"Faithfulness Score: {result}")

Faithfulness Score: MetricResult(value=0.0)


In [20]:
# Evaluate score should be 1
result = await metric.ascore(
    user_input="When was the first super bowl?",
    response="The first superbowl was held on Jan 15, 1967",
    retrieved_contexts=[
        "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
    ]
)
print(f"Faithfulness Score: {result.value}")

Faithfulness Score: 1.0


In [ ]:
#test
response=client.chat.completions.create(model=LLM_MODEL,messages=[{"role":"user", "content": "why sky is blue?"}])
response

In [9]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=collection_name,
    url="http://localhost:6333",
    content_payload_key="text",
)

base_retriever = qdrant.as_retriever(search_kwargs={"k" : 2})

In [106]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Answer the question based only on the following context. 
Give short answer.
If you cannot answer the question with the context, please respond with 'I don't know':

### CONTEXT
{context}

### QUESTION
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [71]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

retrieval_augmented_qa_chain = (
    {"context": itemgetter("question") | base_retriever, "question": itemgetter("question")} 
    | RunnablePassthrough.assign(context=lambda x: "\n---\n".join([c.page_content for c in x["context"]]))
    | {"response": prompt | llm, "context" : itemgetter("context") }
)

In [72]:
q='What does Plato criticize about the practice of medicine in his day?'
res=retrieval_augmented_qa_chain.invoke({"question" : q})
print(res)

{'response': 'Answer:\n\nPlato criticizes the practice of medicine in his day for several reasons:\n\n1. He disapproves of invalidism interfering with the business of life.\n2. He does not recognize that time is the great healer both of mental and bodily disorders, and prefers remedies that produce a sudden catastrophe over gradual ones.\n3. Plato fails to see the importance of diet in healing and influencing the body through control of eating and drinking.\n4. He does not understand that greater simplicity in medicine would be beneficial for its progress.\n5. Physicians have focused too much on curing diseases rather than understanding the conditions of health as a whole.\n6. The improvements in medicine have been more than counterbalanced by the disuse of regular training.\n7. Plato criticizes physicians for not giving enough importance to air and water, which were well understood by ancient times.\n8. He opposes the idea of getting rid of invalids and useless lives by leaving them t

In [ ]:
eval_template="""
### TASK
evaluate relevancy and of model generated answer. 
given contexts, question, ground truth and said answer in your evaluation. 
Give score between 1-3, 2 is relevant, 1 not relevant and 3 very relevant.

### CONTEXT
{context}

### QUESTION
{question}

### GROUND TRUTH
{ground_truth}

### ANSWER
{answer}

### OUTPUT
response with this JSON format (literal example):
{{"score": 1,
 "evaluation":"some reasoning"}}
"""

In [25]:
df=pd.read_csv('../data/cleanse_q_a.csv',index_col=["idx"])

In [73]:
from tqdm.auto import tqdm

In [81]:
answers=[]
for i,q in tqdm(enumerate(df.question.to_list()),total=len(df.question.to_list())):
#    print(i,q)
    res=retrieval_augmented_qa_chain.invoke({"question" : q})
    answers.append(res['response'])
    #print(res)

0it [00:00, ?it/s]

In [ ]:
df["answer"]=answers

In [69]:
df["retrieved_contexts"]=retrieved_contexts

In [81]:
df.to_pickle("../data/evaluation_data.pkl")

## Evaluation

In [1]:
import pandas as pd

df=pd.read_pickle("../data/evaluation_data.pkl")
df=df.rename(columns={"question":"user_input","answer":"response","ground_thruth":"reference"})

In [2]:
from IPython.core.display import HTML
HTML(df.head(1).to_html())

In [3]:
from datasets import Dataset 

dataset = Dataset.from_dict(df.iloc[:2])

In [4]:
import os
import nest_asyncio
import logging

from ragas import evaluate

# ⚠️ DO NOT import instructor here or anywhere else in the notebook.

# Import the standard LangChain client and Ragas wrapper
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper # Ensures the 'agenerate_prompt' interface is present

# --- Configuration & Setup ---
#logging.basicConfig(level=logging.DEBUG) 
nest_asyncio.apply()

LOCAL = False
LLM_SERVER='https://openrouter.ai/api/v1'
LLM_API_KEY='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' #os.environ["OPENROUTER_API_KEY"]
LLM_MODEL='x-ai/grok-4.1-fast:free'

# Setup the standard LangChain model
if LOCAL:
    # Use the LangChain client pointing to your local server
    lc_llm = ChatOpenAI(
        base_url="http://localhost:8081/v1",
        api_key="na",
        model="local-model",
        temperature=0
    )
else:
    lc_llm = ChatOpenAI(
        base_url=LLM_SERVER,
        api_key=LLM_API_KEY,
        model=LLM_MODEL,
        temperature=0
    )

# Wrap it explicitly. This is what provides the required async methods for Ragas.
ragas_llm = LangchainLLMWrapper(lc_llm)


/tmp/ipykernel_206505/2699973973.py:40: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(lc_llm)


In [7]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [8]:
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerSimilarity,
    AnswerCorrectness,
)

# --- Metric and Evaluation ---
faithfulness = Faithfulness(llm=ragas_llm)
answer_relevancy = AnswerRelevancy(llm=ragas_llm)
context_precision = ContextPrecision(llm=ragas_llm)
context_recall = ContextRecall(llm=ragas_llm)
#answer_similarity = AnswerSimilarity(embeddings=embeddings)
answer_correctness = AnswerCorrectness(llm=ragas_llm)
    
metrics = [faithfulness,
           answer_relevancy,
           context_precision,
           context_recall,
           #answer_similarity,
           answer_correctness
         ]


results = evaluate(
    dataset=dataset, 
    metrics=metrics, 
    embeddings=embeddings,
    llm=ragas_llm # Pass the wrapped LLM
)

print(results)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.9615, 'answer_relevancy': 0.2160, 'context_precision': 1.0000, 'context_recall': 1.0000, 'answer_correctness': 0.4187}
